# FIN 4600 Lab: Cryptocurrency Volatility and Risk

**Fintech Foundations · Michigan Technological University**

In this lab you use real market data from the free **CoinGecko API** to measure and compare the risk of five cryptocurrencies: Bitcoin, Ethereum, Cardano, Solana and Chainlink. You will calculate volatility, correlation and diversification, Value at Risk, the Sharpe ratio, drawdown and volatility clustering, and then compare crypto with the S&P 500.

**All the code is written for you.** Your job is to **run it, read it, predict what will happen, try small experiments, and explain what the numbers mean.** You do not need to know how to program. The comments in the code (the lines starting with `#`) explain what each line of Python does, so you will also pick up how Python, pandas and Jupyter notebooks work.

| Part | What you will see | Points |
|---|---|---|
| 1 | Downloading data from a web API and turning it into a table | |
| 2 | Prices, growth of $1, and daily returns | |
| 3 | Volatility: daily, annualized, rolling, and a risk ranking | 15 |
| 4 | Correlation and how much diversification helps | 15 |
| 5 | Tail risk (Value at Risk, Expected Shortfall), Sharpe ratio, drawdown | 40 |
| 6 | Volatility clustering and the risk-return picture | 15 |
| 7 | Crypto vs. the S&P 500, and the limits of this analysis | 15 |

### Symbols used in this notebook

* 🔮 **Predict:** decide what you expect *before* you run the next cell.
* 🧪 **Experiment:** change one value in the code where it says so, run the cell again, and record what happened.
* ✍️ **Answer:** a written question that is graded. Double-click the cell to type in it, then press Shift+Enter. **Cite specific numbers from your tables and charts.**
* ✅ / ❌ in the output: a check of an idea passed / did not pass.

*Educational use only. Nothing here is investment advice.*

## How to use a Jupyter notebook (read this first)

* A notebook is a list of **cells**. This cell is a *text* (Markdown) cell. Cells with a grey background are *code* cells.
* Click a code cell and press **Shift+Enter** to run it. The output appears underneath and the cursor moves to the next cell.
* While a cell runs, its left side shows `[*]` (JupyterLab) or a spinning circle (VS Code). When it finishes, it shows a number such as `[4]`: the order in which cells were run.
* Everything you create (variables, tables) lives in the notebook's memory, called the **kernel**. Run cells **from top to bottom**, because later cells use things made by earlier ones.
* If something gets confusing, restart the kernel and run everything again in order. JupyterLab: menu **Kernel → Restart Kernel and Run All Cells**. VS Code: click **Restart**, then **Run All**, in the toolbar at the top of the notebook.
* If a code cell's last line is just a name (like `vol_table`), Jupyter displays it as a table or value.

### Warm-up: Python in one cell

Run the cell below and compare each line with its output. Every idea here is used later in the lab.

In [ ]:
# A line that starts with # is a COMMENT. Python ignores it; it is a note for people reading the code.

# A VARIABLE is a name that holds a value. "=" means "store the value on the right under the name on the left".
coin = "Bitcoin"              # text goes in quotes and is called a STRING
price_yesterday = 98_000.0    # a number; the underscore is only a readable thousands separator
price_today = 100_000.0

# Arithmetic works like a calculator: + - * /, and ** means "to the power of".
daily_return = price_today / price_yesterday - 1

# print(...) is a FUNCTION: a named command. The values inside the parentheses are its inputs ("arguments").
print(coin)

# An F-STRING (note the f before the opening quote) fills in variables inside {curly braces}.
# After a colon you can add a FORMAT: ",.0f" means thousands separators and 0 decimals; ".2%" means a percentage with 2 decimals.
print(f"{coin} went from ${price_yesterday:,.0f} to ${price_today:,.0f}, a return of {daily_return:.2%}")

# A LIST holds several values in order, inside square brackets. Positions start at 0.
coins = ["Bitcoin", "Ethereum", "Solana"]
print(coins[0])               # coins[0] is the first item: Bitcoin

# A DICTIONARY looks up a value by its key, like a phone book: {key: value, key: value}.
tickers = {"Bitcoin": "BTC", "Ethereum": "ETH", "Solana": "SOL"}
print(tickers["Solana"])      # look up the value stored under the key "Solana": SOL

# A FOR LOOP repeats the indented lines once for each item. Looping over a dictionary gives its keys.
for name in tickers:
    print(name, "trades under the symbol", tickers[name])

# "def" DEFINES your own function. The indented lines are its body, and "return" hands back the result.
def simple_return(new_price, old_price):
    return new_price / old_price - 1

# Now use ("call") the function we just defined, with two inputs.
print(f"Return from our own function: {simple_return(price_today, price_yesterday):.2%}")

## Part 0: Setup

The next cell loads the libraries (ready-made toolkits) the lab uses and sets a few lab-wide settings. It also defines a small `check` function that prints ✅ or ❌ to confirm the ideas you will see along the way.

If you see `ModuleNotFoundError`, the notebook is not using this folder's `.venv` kernel; see the README.

In [ ]:
# ---- Load the libraries this lab uses ----
# "import X" loads a library so we can use it; "import X as Y" also gives it a short nickname.
# "from X import Y" loads just one tool, Y, from library X.
import os                              # talks to the operating system; used to read an optional API key
import time                            # lets the program pause, to respect the API's rate limit
from pathlib import Path               # Path represents the location of a file or folder
from statistics import NormalDist      # the normal ("bell curve") distribution, used in Part 5

import numpy as np                     # numerical math: square roots, logarithms, and more
import pandas as pd                    # tables (DataFrames) and single columns (Series): the main tool of this lab
import matplotlib.pyplot as plt        # draws charts
import requests                        # downloads data from web APIs
import seaborn as sns                  # nicer chart styles, plus heatmaps

# ---- Lab settings ----
# Names in CAPITAL letters are a Python convention for settings that stay fixed while the program runs.
BASE_URL = "https://api.coingecko.com/api/v3"   # the start of every CoinGecko web address we request
DAYS = 365              # days of history to download (the free API allows at most 365)
TRADING_DAYS = 365      # crypto trades every day of the year; stock markets are open about 252 days
COINS = {               # a dictionary: the name we display -> the id CoinGecko uses in its web addresses
    "Bitcoin": "bitcoin",
    "Ethereum": "ethereum",
    "Cardano": "cardano",
    "Solana": "solana",
    "Chainlink": "chainlink",
}

# The folder where downloaded data are saved: a folder named "data" inside this lab folder.
DATA_DIR = Path("data")
# Create the folder if it is missing. exist_ok=True means "it is fine if it already exists".
DATA_DIR.mkdir(exist_ok=True)

# Optional: a free CoinGecko "Demo" API key raises the rate limit. Most students will not need one.
# os.getenv reads an environment variable, and gives back None (Python's "nothing") if it is not set.
API_KEY = os.getenv("COINGECKO_API_KEY")
# A one-line "X if test else Y": send the key with each request if we have one, otherwise send nothing extra.
HEADERS = {"x-cg-demo-api-key": API_KEY} if API_KEY else {}

# ---- Display settings ----
# Show decimal numbers in tables with thousands separators and 4 decimals.
# "lambda x: ..." is a tiny unnamed function: it takes a number x and returns it as formatted text.
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")   # a clean chart style with a light grid
plt.rcParams["figure.dpi"] = 110                   # draw charts a little sharper than the default


def check(label, condition):
    """Print a green check if condition is True, or a red cross if it is False."""
    # The text in triple quotes just above is a DOCSTRING: a short description of what the function does.
    # ("✅ " if condition else "❌ ") picks the first symbol when condition is True, the second when it is False.
    print(("✅ " if condition else "❌ ") + label)


# pd.__version__ holds the version number of the pandas library that was loaded.
print("Setup complete. pandas version", pd.__version__)

## Part 1: Getting data from the CoinGecko API

An **API** (Application Programming Interface) is a web address that returns data for programs instead of a web page for people. CoinGecko offers free market data with no account required. We use one endpoint:

```
GET https://api.coingecko.com/api/v3/coins/{id}/market_chart?vs_currency=usd&days=365
```

It returns **JSON** (a text format for lists and dictionaries) with three lists, each made of `[timestamp, value]` pairs:

```python
{"prices":        [[1700006400000, 36500.1], [1700092800000, 37100.9], ...],
 "market_caps":   [[1700006400000, 7.1e11],  ...],
 "total_volumes": [[1700006400000, 1.9e10],  ...]}
```

Two details matter: timestamps count **milliseconds** since 1 January 1970 (UTC), and the last point is *today's partial day*, so two points can fall on the same calendar date.

The next cell defines two functions. `parse_market_chart` turns that JSON into a pandas **DataFrame** (a table with labelled rows and columns). `fetch_crypto_data` asks CoinGecko for one coin and hands the answer to `parse_market_chart`. Defining a function does not run it yet; we use them in the cells after.

In [ ]:
def parse_market_chart(data):
    """Turn CoinGecko's market_chart JSON into a DataFrame with one row per calendar day."""
    # data["prices"] is a list of [timestamp, price] pairs.
    # [p[1] for p in data["prices"]] is a LIST COMPREHENSION: it walks through every pair p
    # and keeps p[1], the second item of the pair (the value). p[0], the first item, is the timestamp.
    df = pd.DataFrame(
        {
            "price": [p[1] for p in data["prices"]],             # column 1: price in US dollars
            "market_cap": [m[1] for m in data["market_caps"]],   # column 2: market capitalization
            "volume": [v[1] for v in data["total_volumes"]],     # column 3: 24-hour trading volume
        },
        # The row labels (the INDEX) are the timestamps, converted from milliseconds into dates and times.
        index=pd.to_datetime([p[0] for p in data["prices"]], unit="ms"),
    )
    # .normalize() moves every time to midnight, so all points from the same day get the same date.
    df.index = df.index.normalize()
    # Name the index "date"; it becomes the column header when the table is saved to a file.
    df.index.name = "date"
    # df.index.duplicated(keep="last") is True for each repeated date except its last appearance.
    # "~" flips True and False, so df[...] keeps every row that is NOT an earlier duplicate.
    df = df[~df.index.duplicated(keep="last")]
    # Hand the finished table back to whoever called the function.
    return df


def fetch_crypto_data(coin_id, days=DAYS):
    """Download daily USD price, market cap and volume for one coin from CoinGecko."""
    # days=DAYS gives the input a DEFAULT value, used whenever the caller leaves it out.
    # Build the web address, for example https://api.coingecko.com/api/v3/coins/bitcoin/market_chart
    url = f"{BASE_URL}/coins/{coin_id}/market_chart"
    # The query PARAMETERS that go after the "?" in the address: prices in US dollars, for this many days.
    params = {"vs_currency": "usd", "days": days}
    # Send the request. timeout=30 means give up after 30 seconds instead of waiting forever.
    response = requests.get(url, params=params, headers=HEADERS, timeout=30)
    # If the server answered with an error (for example 429, "too many requests"), stop here and report it.
    response.raise_for_status()
    # .json() converts the JSON text into Python lists and dictionaries; then build the table.
    return parse_market_chart(response.json())

### See the parser work on a tiny example

Before using real data, feed `parse_market_chart` a small made-up answer in exactly CoinGecko's format: four data points, where the last two fall on the same day.

🔮 **Predict:** four data points go in. How many rows will the table have?

In [ ]:
# A tiny made-up API answer. 1e9 is scientific notation: 1 × 10^9 = 1,000,000,000.
sample = {
    "prices":        [[1700006400000, 100.0], [1700092800000, 110.0], [1700179200000, 121.0], [1700200000000, 125.0]],
    "market_caps":   [[1700006400000, 1e9],   [1700092800000, 1.1e9], [1700179200000, 1.2e9], [1700200000000, 1.3e9]],
    "total_volumes": [[1700006400000, 5e6],   [1700092800000, 6e6],   [1700179200000, 7e6],   [1700200000000, 8e6]],
}

# What do the raw timestamps mean? Loop over the price pairs.
# "for stamp, price in ..." UNPACKS each two-item pair into two variables at once.
for stamp, price in sample["prices"]:
    # pd.to_datetime(..., unit="ms") converts milliseconds since 1970 into a readable date and time.
    print(stamp, "->", pd.to_datetime(stamp, unit="ms"), "  price:", price)

# Run our parser on the sample.
sample_df = parse_market_chart(sample)

# .iloc[-1] means "the item at the last position" (negative positions count from the end).
check("4 data points became 3 rows, one per calendar day", len(sample_df) == 3)
check("for the repeated day, the later point was kept (125, not 121)", sample_df["price"].iloc[-1] == 125.0)

# A bare name on the last line: Jupyter displays the table.
sample_df

### Download the real data

`load_or_fetch` downloads one coin with `fetch_crypto_data` and **caches** (saves) the result as a CSV file in `data/`, so running the notebook again does not call the API again. The free API is rate limited: an HTTP **429** error means "slow down", and the function waits and retries automatically.

If your instructor provided the `data/` files, everything loads instantly and everyone in class analyzes the same data.

In [ ]:
def load_or_fetch(coin_id, days=DAYS, refresh=False, retries=3):
    """Load one coin from the data/ folder if it is there; otherwise download it and save a copy."""
    # The file for this coin, for example data/bitcoin_365d.csv. Here "/" joins a folder and a file name.
    path = DATA_DIR / f"{coin_id}_{days}d.csv"

    # If the file exists and we did not ask for a refresh, read it and return right away.
    # index_col="date" makes the date column the row labels; parse_dates=True reads it as dates, not text.
    if path.exists() and not refresh:
        return pd.read_csv(path, index_col="date", parse_dates=True)

    # Otherwise download it, trying up to "retries" times. range(1, retries + 1) counts 1, 2, 3.
    for attempt in range(1, retries + 1):
        # "try" runs the indented code. If it fails with an HTTPError, Python jumps to "except" instead of stopping.
        try:
            df = fetch_crypto_data(coin_id, days)
            break                               # success, so leave the loop
        except requests.HTTPError as err:
            # The status code of the failed request (429 = too many requests).
            status = err.response.status_code if err.response is not None else None
            if status == 429 and attempt < retries:
                wait = 30 * attempt             # wait 30 seconds, then 60: back off more each time
                print(f"  rate limited on {coin_id}; waiting {wait}s (attempt {attempt}/{retries})")
                time.sleep(wait)
            else:
                raise                           # a different error, or out of retries: show the error

    # Save the table as a CSV file so the next run loads it instantly.
    df.to_csv(path)
    # Pause 2.5 seconds to be polite to the free API before the next download.
    time.sleep(2.5)
    return df


# Load Bitcoin. .index.min() is the earliest date and .index.max() the latest; .date() drops the time of day.
btc = load_or_fetch("bitcoin")
print(f"Bitcoin: {len(btc)} rows, from {btc.index.min().date()} to {btc.index.max().date()}")

# .head() shows the first 5 rows of the table.
btc.head()

## Part 2: Prices and returns

Next we load all five coins and put their prices side by side in one table, `prices`: one **column per coin** and one **row per date**.

In [ ]:
# A DICTIONARY COMPREHENSION builds a dictionary in one line.
# COINS.items() gives (display name, CoinGecko id) pairs; for each pair, store name -> that coin's table.
raw = {name: load_or_fetch(coin_id) for name, coin_id in COINS.items()}

# Take only the "price" column from each coin's table and place them side by side, matched by date.
# .dropna() then drops any date where some coin has no price ("NA" = not available),
# so every column covers exactly the same dates.
prices = pd.DataFrame({name: df["price"] for name, df in raw.items()}).dropna()

# .shape is (number of rows, number of columns).
print("prices table shape:", prices.shape)

# .tail() shows the last 5 rows.
prices.tail()

### Growth of $1

Bitcoin's price is thousands of times Cardano's, so plotting raw prices on one chart is useless. Instead, divide each coin's prices by its first price: every line then starts at $1 and shows what $1 invested on the first day became.

In [ ]:
# prices.iloc[0] is the first ROW (iloc looks things up by position; 0 = first).
# Dividing the whole table by it divides each column by that coin's own first price.
growth = prices / prices.iloc[0]

# plt.subplots() creates an empty figure (fig) and a drawing area (ax). figsize is (width, height) in inches.
fig, ax = plt.subplots(figsize=(11, 5))
growth.plot(ax=ax, linewidth=1.5)                 # one line per column (coin)
ax.axhline(1, color="black", linewidth=0.8)       # a horizontal reference line at $1 (break-even)
ax.set_title("Growth of $1 invested on the first day")
ax.set_ylabel("Value of $1")
ax.set_xlabel("")
plt.show()                                        # display the finished chart

# The last row of growth is what $1 became by the final day. .round(2) keeps 2 decimals.
growth.iloc[-1].round(2)

### Daily returns

Risk is measured on **returns** (percentage changes), not on prices. The **simple daily return** is

$$r_t = \frac{P_t}{P_{t-1}} - 1$$

A **log return**, $\ln(P_t / P_{t-1})$, is almost the same number for small moves and is common in finance research. Here is the idea on three made-up prices, first by hand and then with pandas:

In [ ]:
# pd.Series is a single labelled column of numbers.
mini = pd.Series([100.0, 110.0, 99.0], name="price")

# By hand: 100 -> 110 is +10%; 110 -> 99 is -10%. ":.1%" shows each result as a percentage with 1 decimal.
print(f"By hand: day 2 = {110 / 100 - 1:.1%}   day 3 = {99 / 110 - 1:.1%}")

# .pct_change() computes (this row / previous row) - 1 for every row.
# The first row has no previous row, so it shows NaN ("not a number").
mini.pct_change()

In [ ]:
# The same calculation on every coin at once: pandas applies it to all columns together.
# .dropna() removes the first row, which is empty because day 1 has no previous day.
returns = prices.pct_change().dropna()

# Log returns: ln(today / yesterday). prices.shift(1) slides every column down one row,
# so on each row it holds yesterday's price. np.log is the natural logarithm.
log_returns = np.log(prices / prices.shift(1)).dropna()

# Compare the two kinds of return for Bitcoin on the first day.
# .iloc[0] is the first value; ":.4%" shows a percentage with 4 decimals.
print(f"Bitcoin, first day: simple return {returns['Bitcoin'].iloc[0]:.4%}, log return {log_returns['Bitcoin'].iloc[0]:.4%}")
check("the returns table has exactly one row fewer than prices", len(returns) == len(prices) - 1)

# .describe() summarizes each column: count, mean, standard deviation (std), min, quartiles, max.
# .T TRANSPOSES (flips) the result so each coin is a row.
returns.describe().T

**Look at the table above.** Compare each coin's `std`, `min` and `max`. Which coins swing the most? Part 3 puts numbers on this.

## Part 3: Measuring volatility (15 points)

**Volatility** is the standard deviation of returns. To turn *daily* volatility into an *annual* figure, multiply by $\sqrt{N}$, where $N$ is the number of return observations per year. Crypto trades 365 days a year, so here $N = 365$ (for stocks, about 252).

Why a square root? If daily returns are independent, *variances* add up over days, so the variance of a year is $N$ times the daily variance, and the standard deviation is $\sqrt{N}$ times the daily one.

In [ ]:
def volatility_metrics(r, periods=TRADING_DAYS):
    """Volatility statistics for one Series of daily returns r."""
    r = r.dropna()                    # ignore any missing values
    daily = r.std()                   # standard deviation of daily returns = daily volatility
    # Return a DICTIONARY of results. Each key becomes a column of the table below.
    return {
        "daily_vol": daily,
        "annual_vol": daily * np.sqrt(periods),   # annualize: multiply by the square root of days per year
        "worst_day": r.min(),                     # the single worst daily return
        "best_day": r.max(),                      # the single best daily return
        "range": r.max() - r.min(),               # distance from the worst day to the best day
    }


# Run the function once per coin (returns.columns is the list of coin names) and collect the results.
# pd.DataFrame(...) puts each coin in a COLUMN; .T flips the table so each coin is a ROW.
vol_table = pd.DataFrame({name: volatility_metrics(returns[name]) for name in returns.columns}).T

print(f"√365 = {np.sqrt(365):.2f}, so annual volatility is about 19 times daily volatility.")
# np.allclose checks that two sets of numbers are equal apart from tiny rounding differences.
check("annual_vol = daily_vol × √365 for every coin", np.allclose(vol_table["annual_vol"], vol_table["daily_vol"] * np.sqrt(365)))
vol_table

### Rolling volatility

A single number for the whole year hides how risk changes over time. **Rolling volatility** recomputes the standard deviation over a moving window, here the last 30 days, for every date.

🔮 **Predict:** will each coin's volatility stay roughly constant through the year, or rise and fall? Will the coins rise and fall at the same times?

In [ ]:
# 🧪 EXPERIMENT (Q1b): change 30 to 7, run this cell, then try 90. Set it back to 30 when you are done.
WINDOW = 30

# .rolling(window=WINDOW) looks at a moving block of WINDOW consecutive days; .std() takes the standard
# deviation inside each block. Multiplying by √365 annualizes it.
# The first WINDOW - 1 rows are NaN, because a full block of days is not yet available.
rolling_vol = returns.rolling(window=WINDOW).std() * np.sqrt(TRADING_DAYS)

fig, ax = plt.subplots(figsize=(11, 5))
rolling_vol.plot(ax=ax, linewidth=1.5)
ax.set_title(f"{WINDOW}-day rolling annualized volatility")
ax.set_ylabel("Annualized volatility")
ax.set_xlabel("")
# Show the y-axis tick labels as percentages. FuncFormatter applies a small function to every label;
# "_" is a conventional name for an input we do not use.
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="")
plt.show()

### Rank the coins by risk

Sort the coins from least to most volatile and express each one's volatility as a multiple of Bitcoin's.

In [ ]:
# Sort the rows by annual volatility, smallest first.
# .copy() makes an independent table, so adding a column below does not change vol_table.
ranking = vol_table.sort_values("annual_vol").copy()

# A new column: each coin's annual volatility divided by Bitcoin's.
# .loc["Bitcoin", "annual_vol"] looks up a single cell by its row label and column label.
ranking["x_bitcoin"] = ranking["annual_vol"] / vol_table.loc["Bitcoin", "annual_vol"]

fig, ax = plt.subplots(figsize=(8, 4))
# A horizontal bar chart (barh) of one column. sns.color_palette("deep")[0] is the palette's first color.
ranking["annual_vol"].plot.barh(ax=ax, color=sns.color_palette("deep")[0])
ax.set_title("Annualized volatility, least to most volatile")
ax.set_xlabel("Annualized volatility")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.show()

ranking

✍️ **Q1a (10 pts).** Which coin was the most volatile and which was the least? Give their annualized volatilities and how many times Bitcoin's volatility the riskiest coin had. What features of these assets (size, age, use case, liquidity) might explain the ranking?

*Your answer:*

✍️ **Q1b (5 pts).** 🧪 Describe the rolling volatility chart with a 7-day and a 90-day window compared with 30 days. What is the trade-off in choosing the window length? Did the coins' volatility rise and fall at the same times?

*Your answer:*

## Part 4: Correlation and diversification (15 points)

If assets do not move together, combining them lowers risk. The **correlation** of two return series measures how much they move together: +1 = perfectly together, 0 = unrelated, −1 = opposite. The **correlation matrix** shows it for every pair of coins.

In [ ]:
# .corr() computes the correlation of every pair of columns: a 5 × 5 table.
corr = returns.corr()

# The average of the OFF-diagonal correlations. The diagonal holds 5 ones (each coin with itself),
# which would inflate the average, so: (sum of all 25 cells - 5) / (25 - 5 off-diagonal cells).
n = len(corr)                                          # the number of coins: 5
avg_pairwise_corr = (corr.values.sum() - n) / (n * n - n)

fig, ax = plt.subplots(figsize=(7, 5.5))
# A heatmap colors each cell by its value. annot=True writes the number in the cell; fmt=".2f" uses 2 decimals.
# cmap is the color scale (red = low, green = high); vmin and vmax fix the scale to the full range -1 to +1.
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlation of daily returns")
plt.show()

print(f"Average pairwise correlation: {avg_pairwise_corr:.2f}")

### A portfolio of all five coins

Now hold all five coins, 20% in each, rebalanced daily back to those weights. The portfolio's daily return is the weighted sum of the coins' returns.

🔮 **Predict:** will the portfolio's volatility be *above*, *equal to*, or *below* the average volatility of the five coins? By a lot or a little?

In [ ]:
# 🧪 EXPERIMENT (Q2b): change the weights (they must add up to 1.0). Try all 1.0 in Bitcoin and 0 elsewhere,
# then 0.5 Bitcoin and 0.5 Ethereum. Set them back to 0.2 each when you are done.
WEIGHTS = {"Bitcoin": 0.2, "Ethereum": 0.2, "Cardano": 0.2, "Solana": 0.2, "Chainlink": 0.2}

# Turn the dictionary into a pandas Series, labelled by coin name, so it lines up with the columns of returns.
weights = pd.Series(WEIGHTS)
check(f"the weights add up to 1 (they add up to {weights.sum():.2f})", np.isclose(weights.sum(), 1.0))

# returns * weights multiplies each coin's column by that coin's weight.
# .sum(axis=1) then adds across each ROW (axis=1 means "across the columns"): one portfolio return per day.
portfolio_returns = (returns * weights).sum(axis=1)

# The portfolio's annualized volatility, with the same formula as for a single coin.
portfolio_annual_vol = portfolio_returns.std() * np.sqrt(TRADING_DAYS)

# The weighted average of the coins' own volatilities: the portfolio's risk if correlations were all +1.
avg_individual_vol = (vol_table["annual_vol"] * weights).sum()

# How much lower the portfolio's volatility is, as a fraction of the average.
benefit = 1 - portfolio_annual_vol / avg_individual_vol

print(f"Portfolio volatility                     : {portfolio_annual_vol:.1%}")
print(f"Weighted average of the coins' volatility: {avg_individual_vol:.1%}")
print(f"Diversification benefit                  : {benefit:.1%} lower")

✍️ **Q2a (10 pts).** Report the average pairwise correlation, the portfolio's volatility, and the weighted average of the coins' volatilities. Does holding all five coins provide strong diversification? Why or why not?

*Your answer:*

✍️ **Q2b (5 pts).** 🧪 What happened to the portfolio's volatility and the diversification benefit with 100% Bitcoin, and with 50% Bitcoin / 50% Ethereum? Explain why.

*Your answer:*

## Part 5: Tail risk and risk-adjusted return (40 points)

### Value at Risk and Expected Shortfall

**Value at Risk (VaR)** answers: *"On a bad day, how much could I lose?"* The 95% one-day historical VaR is the loss that was exceeded on only 5% of days. **Expected Shortfall (ES)** is the *average* loss on those worst 5% of days, so it says how bad the bad days were.

We report both as **positive numbers**: a VaR of `0.05` means a 5% loss.

We also compute VaR the way many simple models do, by *assuming returns follow a normal (bell-curve) distribution*: $\text{VaR} = -(\mu + z\sigma)$, where $\mu$ and $\sigma$ are the mean and standard deviation of daily returns and $z$ is the normal distribution's cutoff (about −2.33 for 99%).

In [ ]:
def historical_var(r, level=0.95):
    """Historical VaR: the loss exceeded on (1 - level) of days, as a positive number."""
    # .quantile(0.05) is the return that 5% of days fell below. The minus sign turns that loss into a positive number.
    return -r.quantile(1 - level)


def expected_shortfall(r, level=0.95):
    """Expected Shortfall: the average loss on the days at or beyond the VaR cutoff."""
    cutoff = r.quantile(1 - level)          # the same cutoff return used for VaR
    # r[r <= cutoff] keeps only the days at or below the cutoff (a FILTER); .mean() averages them.
    return -r[r <= cutoff].mean()


def normal_var(r, level=0.95):
    """VaR assuming returns are normally distributed with the same mean and standard deviation."""
    # inv_cdf(0.01) is the point with 1% of a standard normal distribution below it: about -2.33.
    z = NormalDist().inv_cdf(1 - level)
    return -(r.mean() + z * r.std())


# Build a table: one row per coin, one column per measure.
# Each {c: ... for c in returns} is a dictionary comprehension over the coin names (looping over a DataFrame gives its columns).
var_table = pd.DataFrame(
    {
        "VaR 95%": {c: historical_var(returns[c], 0.95) for c in returns},
        "VaR 99%": {c: historical_var(returns[c], 0.99) for c in returns},
        "ES 95%": {c: expected_shortfall(returns[c], 0.95) for c in returns},
        "Normal VaR 99%": {c: normal_var(returns[c], 0.99) for c in returns},
    }
)

# Check the definitions on Bitcoin. (r_btc < -VaR) is True/False for each day; .mean() of True/False is the share of True.
r_btc = returns["Bitcoin"]
share_worse = (r_btc < -historical_var(r_btc, 0.95)).mean()
check(f"about 5% of Bitcoin's days were worse than its 95% VaR (actual: {share_worse:.1%})", abs(share_worse - 0.05) < 0.01)
check("Expected Shortfall is at least as large as VaR, for every coin", (var_table["ES 95%"] >= var_table["VaR 95%"]).all())

# .style.format("{:.2%}") displays every number as a percentage with 2 decimals.
var_table.style.format("{:.2%}")

**Dollar view.** On the worst 5% of days, a $10,000 position in each coin would have lost at least this much:

In [ ]:
# Multiply the 95% VaR by $10,000, round to whole dollars, give the column a readable name,
# turn the Series into a one-column table (.to_frame()) and show it with dollar signs and commas.
(var_table["VaR 95%"] * 10_000).round(0).rename("Loss on a $10,000 position (95% VaR)").to_frame().style.format("${:,.0f}")

### Are crypto returns normal?

The normal-distribution VaR is only as good as the bell-curve assumption. The next chart compares a histogram of actual daily returns with a normal curve of the same mean and standard deviation.

🔮 **Predict:** compared with the bell curve, will actual returns have *more* or *fewer* very large moves (the "tails" at the far left and right)?

In [ ]:
# 🧪 You may change the coin name to any coin in COINS and run the cell again.
COIN_FOR_HIST = "Bitcoin"
r = returns[COIN_FOR_HIST]

fig, ax = plt.subplots(figsize=(10, 4.5))
# A histogram sorts the returns into 60 bins and draws a bar for each. density=True scales the bars so their
# total area is 1, like a probability curve. alpha=0.6 makes the bars partly see-through.
ax.hist(r, bins=60, density=True, alpha=0.6, label="actual daily returns")

# A normal distribution with the same mean and standard deviation as the actual returns.
normal = NormalDist(r.mean(), r.std())
# np.linspace makes 400 evenly spaced x values from the worst to the best day; draw the bell curve over them.
xs = np.linspace(r.min(), r.max(), 400)
ax.plot(xs, [normal.pdf(x) for x in xs], color="black", label="normal curve, same mean and std")

# Mark the 95% historical VaR with a dashed red line (drawn at a negative return, because it is a loss).
ax.axvline(-historical_var(r, 0.95), color="red", linestyle="--", label="95% VaR")
ax.set_title(f"{COIN_FOR_HIST}: daily returns vs. a normal distribution")
ax.set_xlabel("Daily return")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend()
plt.show()


# How fat are the tails? One row per coin.
tails = pd.DataFrame({
    # Excess kurtosis: 0 for a normal distribution, positive when extreme moves are more common.
    "Excess kurtosis": returns.kurt(),
    # The worst day measured in standard deviations below the average day (a "z-score").
    "Worst day (std devs)": (returns.min() - returns.mean()) / returns.std(),
    # Count the days that moved more than 3 standard deviations from the average, up or down.
    # The comparison gives True/False for every day; .sum() counts the Trues.
    "Days beyond 3 std devs": ((returns - returns.mean()).abs() > 3 * returns.std()).sum(),
})
# Under a normal distribution only about 0.27% of days move more than 3 standard deviations.
# NormalDist().cdf(-3) is the share below -3; double it for both tails, then multiply by the number of days.
tails["Expected if normal"] = len(returns) * 2 * NormalDist().cdf(-3)

tails.style.format("{:.2f}")

✍️ **Q3 (15 pts).** Compare `VaR 99%` with `Normal VaR 99%`, and `ES 95%` with `VaR 95%`. Then use the histogram and the tails table (kurtosis, worst day in standard deviations, days beyond 3 standard deviations). What do they tell you about the shape of crypto return distributions? Why can relying on a normal-distribution VaR be dangerous?

*Your answer:*

### Sharpe ratio

Is the extra risk paid for? The **Sharpe ratio** measures excess return per unit of risk:

$$\text{Sharpe} = \frac{\bar r \cdot N - r_f}{\sigma \cdot \sqrt{N}}, \qquad N = 365$$

where $\bar r$ and $\sigma$ are the mean and standard deviation of *daily* returns and $r_f$ is the annual risk-free rate (what a Treasury bill pays). A negative Sharpe ratio means the asset did worse than the risk-free investment.

In [ ]:
# 🧪 EXPERIMENT (Q4b): change the risk-free rate to 0.05 (5%) and run this cell again. Set it back to 0.02 afterward.
RISK_FREE = 0.02


def sharpe_ratio(r, rf=RISK_FREE, periods=TRADING_DAYS):
    """Annualized Sharpe ratio from a Series of daily returns r."""
    r = r.dropna()
    annual_return = r.mean() * periods            # average daily return × 365
    annual_vol = r.std() * np.sqrt(periods)       # daily volatility × √365
    return (annual_return - rf) / annual_vol      # excess return per unit of risk


# One row per coin: annual return, annual volatility, and Sharpe ratio (passing RISK_FREE explicitly).
# sort_values(..., ascending=False) puts the best Sharpe ratio at the top.
sharpe_table = pd.DataFrame({
    "Annual return": returns.mean() * TRADING_DAYS,
    "Annual vol": vol_table["annual_vol"],
    "Sharpe": pd.Series({c: sharpe_ratio(returns[c], RISK_FREE) for c in returns}),
}).sort_values("Sharpe", ascending=False)

print(f"Risk-free rate used: {RISK_FREE:.0%}")
# A dictionary in .format() gives each column its own display format.
sharpe_table.style.format({"Annual return": "{:.1%}", "Annual vol": "{:.1%}", "Sharpe": "{:.2f}"})

✍️ **Q4a (10 pts).** Which coin had the best Sharpe ratio and which the worst? Is the coin with the highest return also the one with the best Sharpe ratio? What does a negative Sharpe ratio mean?

*Your answer:*

✍️ **Q4b (5 pts).** 🧪 What happened to each Sharpe ratio when the risk-free rate rose from 2% to 5%? Did every coin's Sharpe ratio fall by the same amount? Why or why not?

*Your answer:*

### Drawdown

**Drawdown** is the percentage fall from the highest price reached so far:

$$\text{drawdown}_t = \frac{P_t}{\max_{s \le t} P_s} - 1$$

The **maximum drawdown** is the worst peak-to-trough fall during the period, the loss an unlucky investor who bought at the top would have sat through.

In [ ]:
# prices.cummax() is the running maximum: for each date, the highest price reached so far.
# Dividing today's price by it and subtracting 1 gives the drawdown (0 at a new high, negative below it).
drawdown = prices / prices.cummax() - 1

# The worst (most negative) drawdown of each coin.
max_dd = drawdown.min()

# The gain needed to get back to the old peak: after a fall of d, you need 1 / (1 + d) - 1.
# For example, after a 50% fall (d = -0.5) you need +100% to recover.
recovery_needed = 1 / (1 + max_dd) - 1

fig, ax = plt.subplots(figsize=(11, 4))
drawdown.plot(ax=ax, linewidth=1.2)
ax.set_title("Drawdown from the running peak")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="", ncol=5, loc="lower left")       # legend in 5 columns at the lower left
plt.show()

check("drawdown is never above zero", (drawdown <= 1e-12).all().all())

# Combine the two Series into a table with named columns and show them as percentages.
pd.DataFrame({"Max drawdown": max_dd, "Gain needed to recover": recovery_needed}).style.format("{:.1%}")

✍️ **Q5 (10 pts).** Look at the maximum drawdowns and the gain needed to recover. Why might an investor care more about drawdown than about annualized volatility? Which coin would have been hardest to hold through, and why?

*Your answer:*

### Risk-return summary

This table pulls the results from Parts 3 to 5 into one place, in the original coin order.

In [ ]:
summary = pd.DataFrame({
    "Annual return": returns.mean() * TRADING_DAYS,
    "Annual vol": vol_table["annual_vol"],
    "Sharpe": pd.Series({c: sharpe_ratio(returns[c], RISK_FREE) for c in returns}),
    "VaR 95% (1d)": var_table["VaR 95%"],
    "ES 95% (1d)": var_table["ES 95%"],
    "Max drawdown": max_dd,
}).loc[list(COINS)]          # .loc[list of row labels] puts the rows in the order of the COINS dictionary

summary.style.format({"Annual return": "{:.1%}", "Annual vol": "{:.1%}", "Sharpe": "{:.2f}",
                      "VaR 95% (1d)": "{:.2%}", "ES 95% (1d)": "{:.2%}", "Max drawdown": "{:.1%}"})

## Part 6: Volatility clustering and the risk-return picture (15 points)

### Does volatility cluster?

Financial returns are almost impossible to predict from yesterday's return, but their **size** is predictable: big moves tend to follow big moves, and calm days follow calm days. You can see this with **autocorrelation**: the correlation of a series with a copy of itself shifted by a number of days (the **lag**). A lag-1 autocorrelation compares each day with the day before.

We compare the autocorrelation of the **returns** (direction and size) with that of the **absolute returns** (size only, ignoring the sign: a simple stand-in for volatility).

🔮 **Predict:** which of the two will show clearly positive autocorrelation?

In [ ]:
# 🧪 EXPERIMENT (Q6b): change "Bitcoin" to another coin name from COINS, for example "Solana", and run again.
COIN = "Bitcoin"

# range(1, 11) counts 1, 2, ..., 10 (the end number is not included); list(...) stores those lags in a list.
lags = list(range(1, 11))

# For each lag k, the correlation of the series with itself shifted k days.
acf_returns = [returns[COIN].autocorr(lag=k) for k in lags]          # raw returns
acf_abs = [returns[COIN].abs().autocorr(lag=k) for k in lags]        # .abs() = absolute value (sign removed)

# Two charts side by side: plt.subplots(1, 2) makes 1 row of 2 drawing areas, stored in the list "axes".
# sharey=True gives both charts the same vertical scale so they can be compared directly.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
axes[0].bar(lags, acf_returns, color=sns.color_palette("deep")[0])
axes[0].set_title(f"{COIN}: autocorrelation of returns")
axes[1].bar(lags, acf_abs, color=sns.color_palette("deep")[3])
axes[1].set_title(f"{COIN}: autocorrelation of |returns|")
# Apply the same finishing touches to both charts with a loop.
for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("lag (days)")
    ax.set_xticks(lags)
plt.show()

# np.mean averages the 10 autocorrelations in each list.
print(f"Average autocorrelation, lags 1-10:  returns {np.mean(acf_returns):+.3f}   |returns| {np.mean(acf_abs):+.3f}")

✍️ **Q6a (10 pts).** Describe the two autocorrelation charts. Why is it significant that the autocorrelation of absolute returns is larger than that of raw returns? How could a risk manager use this fact?

*Your answer:*

✍️ **Q6b (5 pts).** 🧪 Which other coin did you try? Did it show the same pattern? Cite the two average autocorrelations printed below the chart.

*Your answer:*

### The risk-return picture

The last chart puts it together: each coin's volatility (risk) on the x-axis, its return on the y-axis, colored by Sharpe ratio. In a world where risk is paid for, points would rise from left to right.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
# A scatter plot: x = volatility, y = return, c = color by Sharpe ratio using the "viridis" color scale.
# s is the marker size; edgecolor draws a black outline around each point.
sc = ax.scatter(summary["Annual vol"], summary["Annual return"], c=summary["Sharpe"],
                cmap="viridis", s=160, edgecolor="black")

# Label each point with its coin's name. .iterrows() loops over the table one row at a time,
# giving the row label (name) and the row's values (row).
for name, row in summary.iterrows():
    # xytext=(8, 6) with textcoords="offset points" places the label 8 points right and 6 points up from the dot.
    ax.annotate(name, (row["Annual vol"], row["Annual return"]), textcoords="offset points", xytext=(8, 6))

ax.axhline(0, color="gray", linewidth=0.8)      # the zero-return line
ax.margins(0.18)                                 # extra space around the points so labels fit
fig.colorbar(sc, ax=ax, label="Sharpe ratio")    # the color legend on the right
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("Annualized return")
ax.set_title("Risk vs. return (color = Sharpe ratio)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.show()

## Part 7: Crypto vs. the S&P 500 (15 points)

How does crypto risk compare with the stock market? The next cell downloads the S&P 500 ETF (**SPY**) for the same dates with the `yfinance` library. If you are offline it skips itself.

Stock markets are open only about **252** days a year, so SPY's daily volatility is annualized with $\sqrt{252}$, not $\sqrt{365}$.

In [ ]:
# "try" attempts the download; if anything goes wrong (no internet, site changed), "except" runs instead,
# so the notebook keeps going.
import warnings

try:
    import yfinance as yf
    # yfinance prints technical warnings meant for its own developers. "with warnings.catch_warnings():"
    # hides them only for the indented block, so the output stays readable.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Download SPY's daily prices from the first to the last date in our crypto data.
        # pd.Timedelta(days=1) adds one day because the "end" date itself is not included.
        # auto_adjust=True adjusts for dividends; ["Close"] keeps the closing price; .squeeze() turns it into a Series.
        spy_prices = yf.download("SPY", start=prices.index[0], end=prices.index[-1] + pd.Timedelta(days=1),
                                 auto_adjust=True, progress=False, threads=False)["Close"].squeeze()
    spy_returns = spy_prices.pct_change().dropna()
    SPY_OK = len(spy_returns) > 100            # True if we got a reasonable amount of data
except Exception as err:
    SPY_OK = False
    print("SPY data unavailable, skipping Part 7:", type(err).__name__)

if SPY_OK:
    # Annualize with √252 (trading days). Also show the WRONG √365 version to see the difference.
    spy_annual_vol = spy_returns.std() * np.sqrt(252)
    spy_wrong_vol = spy_returns.std() * np.sqrt(365)
    print(f"SPY: {len(spy_returns)} trading days (vs {len(returns)} crypto days over the same period)")
    print(f"SPY annual volatility with √252: {spy_annual_vol:.1%}   (with √365, wrongly: {spy_wrong_vol:.1%})\n")

    # Add SPY to the list of crypto volatilities and sort from least to most volatile.
    # pd.concat joins two Series end to end.
    comparison = pd.concat([summary["Annual vol"], pd.Series({"S&P 500 (SPY)": spy_annual_vol})]).sort_values()
    # .apply runs a small function on every value; here it formats each number as a percentage.
    print(comparison.apply(lambda v: f"{v:.1%}").to_string())

    ratio = summary.loc["Bitcoin", "Annual vol"] / spy_annual_vol
    print(f"\nBitcoin was {ratio:.1f}x as volatile as the S&P 500 over this period.")

✍️ **Q7 (15 pts).** (a) How does Bitcoin's volatility compare with SPY's? Why must we annualize with 252 for SPY but 365 for crypto? (b) List **two limitations** of this whole analysis. (Think about sample length, the period chosen, which coins were picked, the data source, and the assumptions behind VaR and Sharpe.)

*Your answer:*

## Submit

1. Restart the kernel and run all cells (JupyterLab: **Kernel → Restart Kernel and Run All Cells**; VS Code: **Restart**, then **Run All**). Every cell should run without errors. Your 🧪 experiment values can be back at their originals, as long as your written answers record what you saw.
2. Make sure every ✍️ answer is filled in and **cites specific numbers**, then save (Ctrl+S or ⌘+S).
3. Submit `crypto_volatility_lab.ipynb` **with outputs visible** as directed in Canvas.

| Part | Questions | Points |
|---|---|---|
| 3 | Q1a, Q1b | 15 |
| 4 | Q2a, Q2b | 15 |
| 5 | Q3, Q4a, Q4b, Q5 | 40 |
| 6 | Q6a, Q6b | 15 |
| 7 | Q7 | 15 |
| | **Total** | **100** |

*Data: CoinGecko API (free tier) and Yahoo Finance via yfinance. Prices are aggregated across exchanges and are for education only; nothing here is investment advice.*